# アニメストーリー構造類似度（アプローチ1）Google Colab 実行ノートブック

このノートブックは、dアニメストア配信アニメ（全体あらすじ欠損を除外した全6,267件）を対象に、
1. **Gemini API** による全話あらすじのプロット構造化要約（起承転結・動機・葛藤・結末・テーマ）の生成
2. **EmbeddingGemma 2 (GPU)** による世界観＆ストーリー展開のデュアルベクトル化
3. **ハイブリッド上位候補プール（Top50 ∪ Top50）** の高速抽出
を一括実行し、Webアプリ用の成果物ZIPファイルを生成します。

---

## 1. GPUの確認 ＆ 依存ライブラリのインストール
※Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で **GPU (T4 または A100)** が選択されていることを確認してください。

In [ ]:
!nvidia-smi
!pip uninstall -y -q torchvision
!pip install -q -U sentence-transformers transformers google-genai


## 2. Google Drive のマウント（推奨）
Google Drive をマウントしておくと、進捗チェックポイント（`checkpoint_summaries.json`）がDrive上に常時保存され、実行が途中で切断されても自動で続きから再開できます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Gemini API Key の設定
Google AI Studio ( https://aistudio.google.com/ ) で取得した API キーを設定します。
- **2つのアカウント** のキーを入力すると、交互利用によりレート制限を回避しながら高速処理できます。
- 1つのみ利用する場合は、アカウント2を空欄（Enter）でスキップしてください。
- （Colab左側の鍵マーク🔑「シークレット」に `GEMINI_API_KEY` を登録している場合は自動読込されます）

In [ ]:
import os, getpass

sec_key = None
try:
    from google.colab import userdata
    sec_key = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if sec_key:
    os.environ["GEMINI_API_KEY"] = sec_key.strip()
    print("✔ Colab シークレットから GEMINI_API_KEY を自動読込しました！")
else:
    api_key_1 = getpass.getpass("Gemini API Key [アカウント1]: ").strip()
    api_key_2 = getpass.getpass("Gemini API Key [アカウント2（任意・空欄でEnter可）]: ").strip()
    keys = [k for k in (api_key_1, api_key_2) if k]
    os.environ["GEMINI_API_KEY"] = ",".join(keys)
    print(f"✔ {len(keys)} 個の API Key を登録しました！")


## 4. 入力CSVファイルの配置確認
2つのCSVファイル（`danime_synopses_1990_2026_animeonly.csv` と `danime_works_with_story_sequence_1990_2026.csv`）のパスを確認します。

In [ ]:
import os

def find_file(filename):
    candidates = [
        f"/content/{filename}",
        f"/content/drive/MyDrive/{filename}",
        f"/content/drive/MyDrive/アニメ類似度/{filename}",
        f"/content/drive/MyDrive/アニメストーリー構造類似度/{filename}",
        f"/content/drive/MyDrive/Annict/{filename}",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return f"/content/{filename}"

SYNOPSIS_CSV = find_file("danime_synopses_1990_2026_animeonly.csv")
SEQUENCE_CSV = find_file("danime_works_with_story_sequence_1990_2026.csv")

os.environ["SYNOPSIS_CSV"] = SYNOPSIS_CSV
os.environ["SEQUENCE_CSV"] = SEQUENCE_CSV

print("全体あらすじCSV:", SYNOPSIS_CSV, "(存在:", os.path.exists(SYNOPSIS_CSV), ")")
print("全話あらすじCSV:", SEQUENCE_CSV, "(存在:", os.path.exists(SEQUENCE_CSV), ")")

if not os.path.exists(SYNOPSIS_CSV) or not os.path.exists(SEQUENCE_CSV):
    print("\n【注意】CSVが見つかりません。左側ファイル欄(/content/)に2つのCSVをドラッグ＆ドロップしてください。")


## 5. 出力先ディレクトリの設定
Google Drive がマウントされている場合は優先して Drive に保存します（途中で切断されても安心です）。

In [ ]:
import os

if os.path.exists("/content/drive/MyDrive"):
    output_dir = "/content/drive/MyDrive/アニメストーリー構造類似度"
else:
    output_dir = "/content/story_structure_output"

os.makedirs(output_dir, exist_ok=True)
os.environ["OUTPUT_DIR"] = output_dir
print(f"✔ 出力先ディレクトリを設定しました: {output_dir}")


## 6. パイプラインの一括実行
`story_structure_similarity_colab.py` を実行します。
- 全体あらすじ欠損（83件）を自動除外し、厳密に6,267件を処理
- Gemini API によるプロット構造化要約（1件ごとにチェックポイント保存・レジューム対応・無料枠最適化）
- EmbeddingGemma 2 による世界観＆プロットのデュアルベクトル化
- ハイブリッド上位候補プール（Top50 ∪ Top50）の抽出
- 成果物ZIPアーカイブの作成

※まずは動作確認したい場合は末尾に `--test` を付けて実行できます。

In [ ]:
# パイプライン実行（デフォルトで最適な軽量モデル・レート制限で安全実行）
!python story_structure_similarity_colab.py


## 7. 生成された成果物ZIPのダウンロード
全件の処理完了後、以下のセルを実行して `story_structure_similarity_results.zip` をPCにダウンロードしてください。

In [ ]:
import os
from google.colab import files

output_dir = os.environ.get("OUTPUT_DIR", "/content/drive/MyDrive/アニメストーリー構造類似度")
zip_candidates = [
    os.path.join(output_dir, "story_structure_similarity_results.zip"),
    "/content/drive/MyDrive/アニメストーリー構造類似度/story_structure_similarity_results.zip",
    "/content/story_structure_output/story_structure_similarity_results.zip",
    "/content/output/story_structure_similarity_results.zip",
]

target_zip = None
for z in zip_candidates:
    if os.path.exists(z):
        target_zip = z
        break

if target_zip:
    print(f"成果物ZIPをダウンロードします: {target_zip} ({os.path.getsize(target_zip) / (1024*1024):.2f} MB)")
    files.download(target_zip)
else:
    print("成果物ZIPがまだ生成されていません。全作品の要約完了後に自動生成されます。")
